### **Multiple Schemas**

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langgraph

#### **Private State**

In [ ]:
from typing_extensions import TypedDict
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END

class PublicState(TypedDict):
    foo: int

class PrivateState(TypedDict):
    baz: int

def node_1(state: PublicState) -> PrivateState:
    print('---Node 1---')
    return {'baz': state['foo'] + 1}

def node_2(state: PrivateState) -> PublicState:
    print('---Node 2---')
    return {'foo': state['baz'] + 1}

# Build graph
builder = StateGraph(PublicState) # Can only receive 'foo' in the output
builder.add_node('node_1', node_1)
builder.add_node('node_2', node_2)
builder.add_node('node_3', node_3)

# Logic
builder.add_edge(START, 'node_1')
builder.add_edge('node_1', 'node_2')
builder.add_edge('node_2', END)

# Add
graph = builder.compile()

# View
display(Image(graph.get_graph().draw_mermaid_png()))

# Invocation
graph.invoke({'foo' : 1})

`baz` is only included in `PrivateState`. `node_2` uses `PrivateState` as input, but writes out to `OverallState`.

So, we can see that `baz` is excluded from the graph output because it is not in `OverallState`.

#### **Input / Output Schema**
By default, a `StateGraph` uses a single schema, so all nodes, inputs, and outputs must follow the same data structure. 

However, you can also define separate input and output schemas:
- The input schema controls what information users provide to the graph.
- The output schema controls what information the graph returns to users.
- The internal schema can contain extra fields used only inside the graph.

This lets you keep your graph’s internal details private, while only exposing or requiring what’s necessary for users.

In [ ]:
class OverallState(TypedDict):
    question: str
    answer: str
    notes: str

def thinking_node(state: OverallState):
    return {'answer': 'bye', 'notes': '... his name is Lance'}

def answer_node(state: OverallState):
    return {'answer': 'bye Lance'}

graph = StateGraph(OverallState)
graph.add_node('answer_node', answer_node)
graph.add_node('thinking_node', thinking_node)
graph.add_edge(START, 'thinking_node')
graph.add_edge('thinking_node', 'answer_node')
graph.add_edge('answer_node', END)

graph = graph.compile()

# View
display(Image(graph.get_graph().draw_mermaid_png()))
graph.invoke({'question' : 'hi'})

In [ ]:
class InputState(TypedDict):
    question: str

class OutputState(TypedDict):
    answer: str

class OverallState(TypedDict):
    question: str
    answer: str
    notes: str

def thinking_node(state: InputState):
    return {'answer': 'bye', 'notes': '... his is name is Lance'}

def answer_node(state: OverallState) -> OutputState:
    return {'answer': 'bye Lance'}

graph = StateGraph(OverallState, input=InputState, output=OutputState)
graph.add_node('answer_node', answer_node)
graph.add_node('thinking_node', thinking_node)
graph.add_edge(START, 'thinking_node')
graph.add_edge('thinking_node', 'answer_node')
graph.add_edge('answer_node', END)

graph = graph.compile()

# View
display(Image(graph.get_graph().draw_mermaid_png()))

graph.invoke({'question': 'hi'})
